# api

> JSON for the Observable Framework frontend (`frontend/`).

In [ ]:
#| default_exp api

In [ ]:
#| hide
from nbdev.showdoc import *

`main` keeps every page and route it already had. This module only adds, beside them:

- `/api/...`: the same operations as JSON, built from `main`'s own functions
- `/app/`: the built frontend (`frontend/dist`), when it exists

and lets the frontend's dev server (<http://localhost:3000>) call the API from another origin.

In [ ]:
#| export
import time
from pathlib import Path
from fasthtml.common import *
from starlette.middleware.cors import CORSMiddleware
from starlette.routing import Mount
from starlette.staticfiles import StaticFiles
from nadayantra import keys
from nadayantra.core import repo_root

In [ ]:
#| export
def mount(app, rt, ns):
    "Add the routes. `ns` is main.py's globals(), so the API uses the very same database, sounds and model."
    sounds, st, SP, G, SECS, nada = (ns[k] for k in 'sounds st SP G SECS nada'.split())
    Sound, blob, vecs, skey, rank, embed, snd, add_ex, newname, typable, typed, lab, cps, hit = (
        ns[k] for k in 'Sound blob vecs skey rank embed snd add_ex newname typable typed lab cps hit'.split())

    app.add_middleware(CORSMiddleware, allow_origin_regex=r'http://(localhost|127\.0\.0\.1):\d+', allow_methods=['*'], allow_headers=['*'])
    dist = repo_root()/'frontend'/'dist'
    if dist.exists(): app.router.routes.insert(0, Mount('/app', app=StaticFiles(directory=dist, html=True), name='frontend'))

    # --- what the page is shown
    def sj(s):
        k = skey(s)
        return dict(id=s.id, name=s.name, glyph=s.glyph, key=s.key, types=k, label=lab(k) if k else '', cps=cps(k) if k else '', n=len(vecs(s)),
                    gk=typed(s.glyph) if s.glyph in G else '')
    def rj(r): return [dict(id=m.id, name=m.name, glyph=m.glyph, cos=float(m.cos), share=float(m.share)) for m in r]
    def hj(h): return [dict(k=x.k, label=lab(x.k), cps=cps(x.k), name=x.name, cos=float(x.cos)) for x in h]

    @rt('/api/state')
    def api_state():
        return dict(nada=dict(state=nada.state, device=nada.device, error=nada.error), sounds=[sj(s) for s in sounds()], sp=SP,
                    mincos=ns['mincos'], hit=hit(), rec=rj(st.rec), hist=hj(st.hist))

    @rt('/api/glyphs')
    def api_glyphs(): return JSONResponse([dict(title=t, glyphs=[dict(sym=g.sym, mean=g.mean, dev=g.dev, iast=g.iast) for g in gs]) for t,gs in SECS])

    @rt('/api/retry')
    def api_retry(): nada.warm_up(); return dict(ok=True)

    # --- teaching and recognising. Each answers {ok, msg}, and the page asks for /api/state again.
    async def learned(audio, f):
        e, err = await embed(audio)
        return dict(ok=False, msg=err) if err else dict(ok=True, msg=f(e))

    @rt('/api/teach')
    async def api_teach(audio:str, key:str='', special:str=''):
        k = special if special in SP else key.strip() if typable(key.strip()) and key.strip() not in SP else ''
        def f(e):
            s = sounds.insert(Sound(name=newname({x.name for x in sounds()}), key=k, samples=blob(e)))
            return f'Saved as “{s.name}”' + (f' (types {lab(k)})' if k else '') + '. Make the same sound again and press Recognize.'
        return await learned(audio, f)

    @rt('/api/learn')
    async def api_learn(audio:str, sym:str):
        def f(e):
            if s := snd(sym): add_ex(s, e); return f'Added another example to {sym} “{s.name}”.'
            s = sounds.insert(Sound(name=G[sym].iast, glyph=sym, samples=blob(e)))
            return f'Saved {sym} as “{s.name}”. Make the same sound again and press Recognize.'
        return await learned(audio, f)

    @rt('/api/more')
    async def api_more(audio:str, id:int):
        def f(e): add_ex(sounds[id], e); return f'Added another example to “{sounds[id].name}”.'
        return await learned(audio, f)

    @rt('/api/recognize')
    async def api_recognize(audio:str):
        if not sounds(): return dict(ok=False, msg='Teach at least one sound first.')
        e, err = await embed(audio)
        if err: return dict(ok=False, msg=err)
        st.rec = rank(e)
        return dict(ok=True, msg='')

    @rt('/api/typekey')
    async def api_typekey(audio:str, sys:bool=False):
        "The sound keyboard. `kind` is how the page should show it; `page` is true when the page itself should type `key`."
        if not any(skey(s) for s in sounds()): return dict(ok=False, kind='warn', msg='No sound has a key yet. Assign keys on the Sound Keyboard page, or teach a glyph.')
        e, err = await embed(audio)
        if err: return dict(ok=False, kind='err', msg=err)
        b = rank(e, keyed=True)[0]
        if b.cos < ns['mincos']: return dict(ok=False, kind='warn', msg=f'Not sure. Closest was “{b.name}” ({b.cos:.2f}), so nothing was typed.')
        k = skey(sounds[b.id])
        st.hist.insert(0, AttrDict(k=k, name=b.name, cos=b.cos)); del st.hist[30:]
        if sys:  # into whichever app has focus
            if keys.type_key(k): return dict(ok=True, kind='ok', page=False, key=k, id=b.id, msg=f'Typed {lab(k)} for “{b.name}” into the active app.')
            return dict(ok=False, kind='err', msg='Typing into other apps is off in macOS. Turn the switch off and on again to fix it.')
        return dict(ok=True, kind='ok', page=True, key=k, id=b.id, msg=f'Typed {lab(k)} for “{b.name}”.')

    # --- editing
    @rt('/api/setkey')
    def api_setkey(id:int, f:str, v:str=''):
        s, v = sounds[id], v.strip()
        if f == 's': k = '' if v == '-' else v
        elif not v: k = s.key if s.key in SP else ''
        else: k = v if typable(v) and v not in SP else s.key  # "up" typed as text is a word, not the Up arrow
        sounds.update(id=id, key=k)
        return dict(ok=True)

    @rt('/api/rename')
    def api_rename(id:int, v:str): sounds.update(id=id, name=v); return dict(ok=True)

    @rt('/api/forget')
    def api_forget(id:int): sounds.delete(id); st.rec = []; return dict(ok=True)

    @rt('/api/forgetall')
    def api_forgetall(): [sounds.delete(s.id) for s in sounds()]; st.rec = []; return dict(ok=True)

    @rt('/api/clear')
    def api_clear(): st.hist.clear(); return dict(ok=True)